# Your first PIEC experiment — local testing
Please download this file and run locally not within the repository root to avoid tracking changes.
Now try sine, square, and ramp waves virtually or on your bench.

**Run the imports cell below first**, then choose where to start:

- **No hardware:** skip to [A. Virtual instruments](#A.-Virtual-instruments).
- **Real hardware:** skip to [B. Physical instruments](#B.-Physical-instruments).

Run **one cell at a time with Shift+Enter** within your chosen section. Each section works independently.

From the repository root, install with `python -m pip install -e . jupyterlab` and launch `python -m jupyter lab`. Physical instruments also need a working VISA backend; see the [connection guide](https://piec.readthedocs.io/en/latest/user_guide/connecting_to_instrument.html).

Prefer a GUI? From the repository root, run `python Measurements/Quickstart/quick_start_local_GUI.py`. It provides the same virtual and Keysight physical examples with settings, plots, and CSV export.

In [ ]:
#Our imports for the entire notebook
import numpy as np
import matplotlib.pyplot as plt

from piec.drivers.awg.virtual_awg import VirtualAwg
from piec.drivers.oscilloscope.virtual_oscilloscope import VirtualScope
from piec.drivers.autodetect import autodetect
from piec.drivers.awg.k_81150a import Keysight81150a
from piec.drivers.oscilloscope.k_dsox3024a import KeysightDSOX3024a

## A. Virtual instruments

Start here without hardware. The first plot shows the applied waveform; the second shows the existing virtual ferroelectric sample response.

In [ ]:
awg = VirtualAwg(simulation_points=1000)
scope = VirtualScope()
print(awg.idn(), "and", scope.idn())

### 2. Start with a sine wave

Set channel 1 to a 1 kHz sine wave. The amplitude is 1 V peak in this simulation.

In [ ]:
frequency_hz = 1_000
amplitude_v = 1.0
#pro tip, hover over the arguments to see what they do, or check the docs for more details
awg.configure_waveform(
    1, "SIN", frequency=frequency_hz,
    amplitude=amplitude_v, offset=0.0,
)

Run the next cell to send the sine wave and see the result.

In [ ]:
applied_voltage = awg.get_waveform(1)
applied_time = np.linspace(0, 1 / frequency_hz, len(applied_voltage))

try:
    awg.output(1, on=True)
    scope.arm()
    awg.send_software_trigger()
    sine_data = scope.get_data()
finally:
    awg.output(1, on=False)

fig, axes = plt.subplots(2, 1, figsize=(8, 5))
axes[0].plot(applied_time * 1_000, applied_voltage)
axes[0].set(title=f"Applied SIN: {frequency_hz:g} Hz",
            xlabel="Time (ms)", ylabel="Applied voltage (V)")
axes[1].plot(sine_data["Time"] * 1_000, sine_data["Voltage"])
axes[1].set(title="Ferroelectric sample response (50 Ω sensing resistor)",
            xlabel="Time (ms)", ylabel="Scope voltage (V)")
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
sine_data.head()

### 3. Try a square wave

Now change only the waveform. The frequency and amplitude stay the same.

In [ ]:
#pro tip, piec is built to be case-insensitive, so 'squ' will also work
awg.set_waveform(1, "SQU")

Run the next cell to send the square wave and see the result.

In [ ]:
applied_voltage = awg.get_waveform(1)
applied_time = np.linspace(0, 1 / frequency_hz, len(applied_voltage))

try:
    awg.output(1, on=True)
    scope.arm()
    awg.send_software_trigger()
    square_data = scope.get_data()
finally:
    awg.output(1, on=False)

fig, axes = plt.subplots(2, 1, figsize=(8, 5))
axes[0].plot(applied_time * 1_000, applied_voltage)
axes[0].set(title=f"Applied SQU: {frequency_hz:g} Hz",
            xlabel="Time (ms)", ylabel="Applied voltage (V)")
axes[1].plot(square_data["Time"] * 1_000, square_data["Voltage"])
axes[1].set(title="Ferroelectric sample response (50 Ω sensing resistor)",
            xlabel="Time (ms)", ylabel="Scope voltage (V)")
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
square_data.head()

### 4. Try a ramp

Now change only the waveform. The frequency and amplitude stay the same.

In [ ]:
awg.set_waveform(1, "RAMP")

Run the next cell to send the ramp and see the result.

In [ ]:
applied_voltage = awg.get_waveform(1)
applied_time = np.linspace(0, 1 / frequency_hz, len(applied_voltage))

try:
    awg.output(1, on=True)
    scope.arm()
    awg.send_software_trigger()
    ramp_data = scope.get_data()
finally:
    awg.output(1, on=False)

fig, axes = plt.subplots(2, 1, figsize=(8, 5))
axes[0].plot(applied_time * 1_000, applied_voltage)
axes[0].set(title=f"Applied RAMP: {frequency_hz:g} Hz",
            xlabel="Time (ms)", ylabel="Applied voltage (V)")
axes[1].plot(ramp_data["Time"] * 1_000, ramp_data["Voltage"])
axes[1].set(title="Ferroelectric sample response (50 Ω sensing resistor)",
            xlabel="Time (ms)", ylabel="Scope voltage (V)")
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
ramp_data.head()

### 5. Save your results

Save each response as a CSV. Rerunning this cell replaces these CSVs.

In [ ]:
sine_data.to_csv("sine.csv", index=False)
square_data.to_csv("square.csv", index=False)
ramp_data.to_csv("ramp.csv", index=False)

**Virtual section complete.** Stop here, or continue below with connected instruments.

For the bench comparison, compare the physical trace with the **applied waveform** above. The virtual response includes a ferroelectric sample; a direct AWG-to-scope connection does not.

## B. Physical instruments

Connect your physical hardware with a standard BNC cable:
1. Connect **AWG channel 1 output → Scope channel 1 input**.
2. Ensure your instruments are powered on and connected to your computer (via USB or GPIB).

### 1. Connect with Autodetect
PIEC includes automatic instrument discovery. Calling `autodetect()` scans your connected instruments and binds the appropriate drivers automatically—no need to look up model names or VISA addresses.

In [ ]:
from piec.drivers.autodetect import autodetect

# Automatically detect and connect to your AWG and oscilloscope
physical_awg = autodetect("awg")
physical_scope = autodetect("scope")

print("Connected:", physical_awg.idn(), "and", physical_scope.idn())

In [ ]:
# --- Alternative: Manual Connection ---
# If autodetect does not find your instrument, or if you prefer manual connection,
# you can pass the instrument's VISA address directly:
# physical_awg = autodetect("GPIB0::10::INSTR")
# physical_scope = autodetect("USB0::0x0957::0x17A6::MY52160123::0::INSTR")
#
# Or import and instantiate your specific model driver directly:
# from piec.drivers.awg.k_81150a import Keysight81150a
# from piec.drivers.oscilloscope.k_dsox3024a import KeysightDSOX3024a
# physical_awg = Keysight81150a("GPIB0::10::INSTR")
# physical_scope = KeysightDSOX3024a("USB0::0x0957::0x17A6::MY52160123::0::INSTR")

### 2. Try a sine wave

Configure channel 1 on the AWG to generate a 1 kHz sine wave.

In [ ]:
physical_frequency_hz = 1_000
physical_amplitude_v = 1.0

physical_awg.configure_waveform(
    1, "SIN", frequency=physical_frequency_hz,
    amplitude=physical_amplitude_v, offset=0.0,
)

Run the next cell to turn on the AWG output, let the scope autoscale to the incoming signal, and capture the waveform.

In [ ]:
try:
    physical_awg.output(1, on=True)
    physical_scope.autoscale()
    physical_sine_data = physical_scope.get_data()
finally:
    physical_awg.output(1, on=False)

plt.figure(figsize=(8, 3))
plt.plot(physical_sine_data["Time"] * 1_000, physical_sine_data["Voltage"])
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (V)")
plt.title("Physical sine wave")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
physical_sine_data.head()

### 3. Try a square wave

Change only the waveform to square ("SQU").

In [ ]:
physical_awg.set_waveform(1, "SQU")

Run the next cell to capture the square wave.

In [ ]:
try:
    physical_awg.output(1, on=True)
    physical_square_data = physical_scope.get_data()
finally:
    physical_awg.output(1, on=False)

plt.figure(figsize=(8, 3))
plt.plot(physical_square_data["Time"] * 1_000, physical_square_data["Voltage"])
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (V)")
plt.title("Physical square wave")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
physical_square_data.head()

### 4. Try a ramp wave

Change the waveform to ramp ("RAMP").

In [ ]:
physical_awg.set_waveform(1, "RAMP")

Run the next cell to capture the ramp wave.

In [ ]:
try:
    physical_awg.output(1, on=True)
    physical_ramp_data = physical_scope.get_data()
finally:
    physical_awg.output(1, on=False)

plt.figure(figsize=(8, 3))
plt.plot(physical_ramp_data["Time"] * 1_000, physical_ramp_data["Voltage"])
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (V)")
plt.title("Physical ramp wave")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
physical_ramp_data.head()

### 5. Save and disconnect

Save your physical results to CSV and safely close instrument sessions.

In [ ]:
physical_sine_data.to_csv("physical_sine.csv", index=False)
physical_square_data.to_csv("physical_square.csv", index=False)
physical_ramp_data.to_csv("physical_ramp.csv", index=False)

# Turn off output and safely close instrument sessions
physical_awg.output(1, on=False)
physical_awg.instrument.close()
physical_scope.instrument.close()
print("Data saved and instruments disconnected safely.")